In [7]:
import numpy as np
import pandas as pd

from pathlib import Path

from sklearn.linear_model import LogisticRegression
from sklearn.metrics import f1_score
from sklearn.preprocessing import StandardScaler

In [8]:
OOF_DIR = Path("../oof")
DATA_DIR = Path("../data")

train_folds = pd.read_csv("train_folds.csv")

In [9]:
convnext_oof = np.load(OOF_DIR / "convnext_oof_logits.npy")
convnext_test = np.load(OOF_DIR / "convnext_test_logits.npy")

swin_oof = np.load(OOF_DIR / "swin_oof_logits.npy")
swin_test = np.load(OOF_DIR / "swin_test_logits.npy")

siglip_oof = np.load(OOF_DIR / "siglip_oof_logits.npy")
siglip_test = np.load(OOF_DIR / "siglip_test_logits.npy")

y_true = np.load(OOF_DIR / "y_true.npy")

In [10]:
print(convnext_oof.shape)
print(swin_oof.shape)
print(siglip_oof.shape)

print(convnext_test.shape)
print(swin_test.shape)
print(siglip_test.shape)

print(y_true.shape)

(24853, 3)
(24853, 3)
(24853, 3)
(1458, 3)
(1458, 3)
(1458, 3)
(24853,)


In [11]:
X_train = np.concatenate(

    [
        convnext_oof,
        swin_oof,
        siglip_oof
    ],

    axis=1

)

X_test = np.concatenate(

    [
        convnext_test,
        swin_test,
        siglip_test
    ],

    axis=1

)

print(X_train.shape)
print(X_test.shape)

(24853, 9)
(1458, 9)


In [12]:
oof_pred = np.zeros(len(y_true), dtype=np.int64)
test_prob = np.zeros((len(X_test), 3))

for fold in sorted(train_folds.fold.unique()):

    train_idx = train_folds.fold != fold
    valid_idx = train_folds.fold == fold

    scaler = StandardScaler()

    X_train_fold = scaler.fit_transform(X_train[train_idx])
    X_valid_fold = scaler.transform(X_train[valid_idx])
    X_test_scaled = scaler.transform(X_test)

    meta = LogisticRegression(
        max_iter=5000,
        random_state=42
    )

    meta.fit(
        X_train_fold,
        y_true[train_idx]
    )

    oof_pred[valid_idx] = meta.predict(X_valid_fold)

    test_prob += (
        meta.predict_proba(X_test_scaled)
        / train_folds.fold.nunique()
    )


In [13]:
score = f1_score(

    y_true,

    oof_pred,

    average="macro"

)

print("Stacking Macro F1 :", score)

Stacking Macro F1 : 0.9824597912160696


In [14]:
test_pred = test_prob.argmax(1)

print(test_pred.shape)

(1458,)


In [15]:
submission = pd.read_csv(

    DATA_DIR / "submission.csv"

)

submission.head()

,id,predicted
0,1,NaN
1,2,NaN
2,3,NaN
3,4,NaN
4,5,NaN


In [16]:
def save_submission(predictions, filename):

    submission = pd.read_csv(DATA_DIR / "submission.csv")

    submission["predicted"] = predictions

    submission.to_csv(
        filename,
        index=False
    )

    print(f"Saved: {filename}")
    print(submission["predicted"].value_counts().sort_index())

In [17]:
save_submission(
    test_pred,
    "submission_stacking.csv"
)

Saved: submission_stacking.csv
predicted
0    515
1    202
2    741
Name: count, dtype: int64


In [18]:
from sklearn.metrics import f1_score

print(
    "ConvNeXt",
    f1_score(
        y_true,
        convnext_oof.argmax(1),
        average="macro"
    )
)

print(
    "Swin",
    f1_score(
        y_true,
        swin_oof.argmax(1),
        average="macro"
    )
)

print(
    "SigLIP",
    f1_score(
        y_true,
        siglip_oof.argmax(1),
        average="macro"
    )
)

ConvNeXt 0.9740210733643745
Swin 0.9748046576395311
SigLIP 0.9604603469912855


In [19]:
convnext_pred = convnext_test.argmax(1)